In [ ]:
import torch
import os
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from transformers import CLIPProcessor, CLIPModel
from tqdm import tqdm

In [ ]:
class CIFAR10WithCLIPEmbeds(Dataset):
    def __init__(self, split="train"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").eval().to(self.device)
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

        self.resize = transforms.Resize((224, 224))
        self.dataset = datasets.CIFAR10(root="./data", train=(split == "train"), download=True, transform=None)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        pil_img, label = self.dataset[idx]
        resized_img = self.resize(pil_img)

        # Prepare input for CLIP
        inputs = self.processor(text=[""], images=resized_img, return_tensors="pt", padding=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            clip_output = self.clip_model(**inputs)
            clip_embed = clip_output.image_embeds.squeeze(0).cpu()  

        return clip_embed, label


In [ ]:
# Hyperparameters
batch_size = 64
num_epochs = 5
learning_rate = 1e-3
num_classes = 10
embedding_dim = 768  # For ViT-L/14

# Dataset & DataLoader
train_dataset = CIFAR10WithCLIPEmbeds(split="train")
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Define Linear Classifier
class LinearClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.fc(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LinearClassifier(embedding_dim, num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

# Training Loop
model.train()
for epoch in range(num_epochs):
    total_loss = 0
    correct = 0
    total = 0

    for embeds, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        embeds, labels = embeds.to(device), labels.to(device)

        logits = model(embeds)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = correct / total * 100
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}, Accuracy = {acc:.2f}%")


In [ ]:
# Prepare test set
test_dataset = CIFAR10WithCLIPEmbeds(split="test")
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Evaluation loop
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for embeds, labels in tqdm(test_loader, desc="Evaluating"):
        embeds, labels = embeds.to(device), labels.to(device)
        logits = model(embeds)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

test_acc = correct / total * 100
print(f"\nTest Accuracy: {test_acc:.2f}%")
